### S3: **Part I - Gene Predcition fro Bacteria**

#### **Utils**

In [1]:
import os, requests
from tqdm import tqdm
import pandas as pd

#### **Data extraction**

In [2]:
!wget -c -P ./Data/ https://ftp.ncbi.nih.gov/genomes/genbank/bacteria/assembly_summary.txt

--2025-01-25 23:49:12--  https://ftp.ncbi.nih.gov/genomes/genbank/bacteria/assembly_summary.txt
Resolving ftp.ncbi.nih.gov (ftp.ncbi.nih.gov)... 130.14.250.10, 130.14.250.11, 130.14.250.31, ...
Connecting to ftp.ncbi.nih.gov (ftp.ncbi.nih.gov)|130.14.250.10|:443... connected.
HTTP request sent, awaiting response... 206 Partial Content
Length: 1154162486 (1.1G), 41723389 (40M) remaining [text/plain]
Saving to: ‘./Data/assembly_summary.txt’

assembly_summary.tx 100%[+++++++++++++++++++>]   1.07G  11.3MB/s    in 3.5s    

2025-01-25 23:49:17 (11.3 MB/s) - ‘./Data/assembly_summary.txt’ saved [1154162486/1154162486]



In [2]:
assembly_summary = pd.read_csv('./Data/assembly_summary.txt', sep='\t', skiprows=1)
assembly_summary.head(5)

/tmp/ipykernel_4125092/3668181553.py:1: DtypeWarning: Columns (5,6,34,35,36) have mixed types. Specify dtype option on import or set low_memory=False.
  assembly_summary = pd.read_csv('./Data/assembly_summary.txt', sep='\t', skiprows=1)


,#assembly_accession,bioproject,biosample,wgs_master,refseq_category,taxid,species_taxid,organism_name,infraspecific_name,isolate,...,replicon_count,scaffold_count,contig_count,annotation_provider,annotation_name,annotation_date,total_gene_count,protein_coding_gene_count,non_coding_gene_count,pubmed_id
0,GCA_036600855.1,PRJNA1051366,SAMN38772065,JBAFXD000000000.1,na,7,7,Azorhizobium caulinodans,strain=CNM20190194,na,...,0.0,171.0,171.0,na,na,NaN,0,0,0,na
1,GCA_036600875.1,PRJNA1051366,SAMN38772066,JBAFXC000000000.1,na,7,7,Azorhizobium caulinodans,strain=CNM20190462,na,...,0.0,171.0,171.0,na,na,NaN,0,0,0,na
2,GCA_036600895.1,PRJNA1051366,SAMN38772064,JBAFXE000000000.1,na,7,7,Azorhizobium caulinodans,strain=CNM20190156,na,...,0.0,260.0,260.0,na,na,NaN,0,0,0,na
3,GCA_036600915.1,PRJNA1051366,SAMN38772067,JBAFXF000000000.1,na,7,7,Azorhizobium caulinodans,strain=CNM20220104,na,...,0.0,135.0,135.0,na,na,NaN,0,0,0,na
4,GCA_019958295.1,PRJNA755163,SAMN20395209,JAIOWO000000000.1,na,9,9,Buchnera aphidicola,strain=Tabriz.4,na,...,0.0,1224.0,1224.0,NCBI,NCBI Prokaryotic Genome Annotation Pipeline (P...,2021-09-08,1179,1050,104,na


In [3]:
assembly_summary.columns

Index(['#assembly_accession', 'bioproject', 'biosample', 'wgs_master',
       'refseq_category', 'taxid', 'species_taxid', 'organism_name',
       'infraspecific_name', 'isolate', 'version_status', 'assembly_level',
       'release_type', 'genome_rep', 'seq_rel_date', 'asm_name',
       'asm_submitter', 'gbrs_paired_asm', 'paired_asm_comp', 'ftp_path',
       'excluded_from_refseq', 'relation_to_type_material',
       'asm_not_live_date', 'assembly_type', 'group', 'genome_size',
       'genome_size_ungapped', 'gc_percent', 'replicon_count',
       'scaffold_count', 'contig_count', 'annotation_provider',
       'annotation_name', 'annotation_date', 'total_gene_count',
       'protein_coding_gene_count', 'non_coding_gene_count', 'pubmed_id'],
      dtype='object')

In [4]:
assembly_summary.assembly_level.unique()

array(['Contig', 'Complete Genome', 'Scaffold', 'Chromosome', '9971912',
       '76'], dtype=object)

In [5]:
assembly_summary[assembly_summary['assembly_level'] == "Complete Genome"].shape

(65397, 38)

In [6]:
assembly_summary.shape

(2487118, 38)

In [7]:
na_count = (assembly_summary['ftp_path'] == "na").sum()
na_count

332

In [8]:
assembly_summary.assembly_level.value_counts()

assembly_level
Contig             2039590
Scaffold            373366
Complete Genome      65397
Chromosome            8763
9971912                  1
76                       1
Name: count, dtype: int64

In [9]:
assembly_summary.assembly_type.value_counts()

assembly_type
haploid    2487116
na               1
Name: count, dtype: int64

In [10]:
assembly_summary.annotation_date.isna().sum()

536392

#### **Focus on annotated data**

In [11]:
annotated_summary = assembly_summary[~assembly_summary['annotation_date'].isna()]
annotated_summary.shape

(1950726, 38)

In [12]:
annotated_summary = annotated_summary[annotated_summary['assembly_level'] == "Contig"]
annotated_summary.shape

(1690257, 38)

In [13]:
annotated_summary.organism_name.unique()

array(['Buchnera aphidicola', 'Dictyoglomus thermophilum',
       'Methylophilus methylotrophus', ...,
       'Campylobacter sp. VTCC 70190', 'Nocardia sp. KC 131',
       'Flavobacterium sp. C3NV'], dtype=object)

In [14]:
len(annotated_summary.organism_name.unique())

43694

In [15]:
count_hyphomonas = (annotated_summary['organism_name'] == 'Hyphomonas sp.').sum()
print(f"Number of rows with organism_name 'Hyphomonas sp.': {count_hyphomonas}")

Number of rows with organism_name 'Hyphomonas sp.': 49


#### **Downloading genome file**

In [16]:
for i, row in annotated_summary.iterrows():
    path = row['ftp_path']
    print(row)
    break

#assembly_accession                                            GCA_019958295.1
bioproject                                                         PRJNA755163
biosample                                                         SAMN20395209
wgs_master                                                   JAIOWO000000000.1
refseq_category                                                             na
taxid                                                                        9
species_taxid                                                                9
organism_name                                              Buchnera aphidicola
infraspecific_name                                             strain=Tabriz.4
isolate                                                                     na
version_status                                                          latest
assembly_level                                                          Contig
release_type                                        

In [18]:
lf = assembly_summary[assembly_summary['asm_name'] == "ASM973039v1"]
lf

,#assembly_accession,bioproject,biosample,wgs_master,refseq_category,taxid,species_taxid,organism_name,infraspecific_name,isolate,...,replicon_count,scaffold_count,contig_count,annotation_provider,annotation_name,annotation_date,total_gene_count,protein_coding_gene_count,non_coding_gene_count,pubmed_id
14511,GCA_009730395.1,PRJNA231221,SAMN11056450,na,reference genome,195,195,Campylobacter coli,strain=FDAARGOS_735,na,...,2.0,2.0,2.0,NCBI,NCBI Prokaryotic Genome Annotation Pipeline (P...,2019-11-22,1760,1663,56,na


In [19]:
lf['assembly_level']

14511    Complete Genome
Name: assembly_level, dtype: object

In [23]:
lf['#assembly_accession'] == "GCA_009730395.1"

14511    True
Name: #assembly_accession, dtype: bool

In [32]:
a = lf.iloc[0]['ftp_path']
a

'https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/009/730/395/GCA_009730395.1_ASM973039v1'

In [16]:
storage_folder = "./Data/RawContig/"
failure_log_path = "./Data/failure.log"
os.makedirs(os.path.dirname(failure_log_path), exist_ok=True)

def log_failure(file_base_name):
    with open(failure_log_path, "a") as log_file:
        log_file.write(f"{file_base_name}\n")

def download_file(url, save_path):
    try:
        response = requests.get(url)
        response.raise_for_status()
        with open(save_path, 'wb') as file:
            file.write(response.content)
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")

<!--
# files to download
    files_to_download = [
        "genomic.fna.gz",
        "genomic.gff.gz",
        
        "ani_contam_ranges.tsv",
        "ani_report.txt",
        "assembly_report.txt",
        "assembly_stats.txt",
        "cds_from_genomic.fna.gz",
        "fcs_report.txt",
        "feature_count.txt.gz",
        "feature_table.txt.gz",
        
        "genomic.gbff.gz",
        
        "genomic.gtf.gz",
        "protein.faa.gz",
        "protein.gpff.gz",
        "rna_from_genomic.fna.gz",
        "translated_cds.faa.gz",
        "wgsmaster.gbff.gz"
    ]
-->

In [ ]:
annotated_summary_ = annotated_summary#.iloc[38526:]
for i, row in tqdm(annotated_summary_.iterrows(), desc="Retreiveing data", nrows=100):
    path = row['ftp_path']
    folder_name = f"{row['#assembly_accession']}_{row['asm_name']}"
    folder_path = os.path.join(storage_folder, folder_name)
    
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    else:
        print(f"Folder '{folder_name}' already exists. Skipping...")
        continue

    # files to download
    files_to_download = [
        "genomic.fna.gz",
        "genomic.gff.gz",
    ]
    for file_name in files_to_download:
        file_base_name = path.split('/')[-1]
        file_url = os.path.join(path, file_base_name) + "_" + file_name
        output_file_path = os.path.join(folder_path, file_name)
        if os.path.exists(output_file_path):
            continue
        try:
            download_file(file_url, output_file_path)
        except Exception as e:
            print(f"Failed to download {file_name}: {e}")
            log_failure(file_base_name)
            if os.path.exists(folder_path) and not os.listdir(folder_path):
                os.rmdir(folder_path)
    #break

Retreiveing data: 1348it [00:01, 1271.58it/s]

Folder 'GCA_019958295.1_ASM1995829v1' already exists. Skipping...
Folder 'GCA_029005255.1_ASM2900525v1' already exists. Skipping...
Folder 'GCA_008244535.1_ASM824453v1' already exists. Skipping...
Folder 'GCA_026418735.1_ASM2641873v1' already exists. Skipping...
Folder 'GCA_039630455.1_ASM3963045v1' already exists. Skipping...
Folder 'GCA_008015785.1_ASM801578v1' already exists. Skipping...
Folder 'GCA_019654935.1_ASM1965493v1' already exists. Skipping...
Folder 'GCA_019655395.1_ASM1965539v1' already exists. Skipping...
Folder 'GCA_033719835.1_ASM3371983v1' already exists. Skipping...
Folder 'GCA_020216695.1_ASM2021669v1' already exists. Skipping...
Folder 'GCA_025397935.1_ASM2539793v1' already exists. Skipping...
Folder 'GCA_031456735.1_ASM3145673v1' already exists. Skipping...
Folder 'GCA_900457065.1_57675_D02' already exists. Skipping...
Folder 'GCA_007197645.1_ASM719764v1' already exists. Skipping...
Folder 'GCA_007991095.1_ASM799109v1' already exists. Skipping...
Folder 'GCA_02319

Retreiveing data: 2699it [00:01, 2800.36it/s]

Folder 'GCA_004806575.1_PDT000381378.1' already exists. Skipping...
Folder 'GCA_004806595.1_PDT000380242.2' already exists. Skipping...
Folder 'GCA_004806655.1_PDT000381382.1' already exists. Skipping...
Folder 'GCA_004806775.1_PDT000382847.1' already exists. Skipping...
Folder 'GCA_004807235.1_PDT000381373.1' already exists. Skipping...
Folder 'GCA_004807255.1_PDT000381372.1' already exists. Skipping...
Folder 'GCA_004807375.1_PDT000381384.1' already exists. Skipping...
Folder 'GCA_004807415.1_PDT000381375.1' already exists. Skipping...
Folder 'GCA_004807455.1_PDT000382629.1' already exists. Skipping...
Folder 'GCA_004807475.1_PDT000382626.1' already exists. Skipping...
Folder 'GCA_004807485.1_PDT000381386.1' already exists. Skipping...
Folder 'GCA_004807535.1_PDT000382630.1' already exists. Skipping...
Folder 'GCA_004807555.1_PDT000382858.1' already exists. Skipping...
Folder 'GCA_004808215.1_PDT000382624.1' already exists. Skipping...
Folder 'GCA_004808235.1_PDT000382631.1' already 

Retreiveing data: 4070it [00:02, 4265.41it/s]

...

Retreiveing data: 115035it [00:18, 6636.99it/s]

Folder 'GCA_045062625.1_PDT002477583.1' already exists. Skipping...
Folder 'GCA_045062645.1_PDT002477581.1' already exists. Skipping...
Folder 'GCA_045062665.1_PDT002477587.1' already exists. Skipping...
Folder 'GCA_045062905.1_PDT002477669.1' already exists. Skipping...
Folder 'GCA_045062975.1_PDT002477578.1' already exists. Skipping...
Folder 'GCA_045063025.1_PDT002477577.1' already exists. Skipping...
Folder 'GCA_045063045.1_PDT002477579.1' already exists. Skipping...
Folder 'GCA_045063065.1_PDT002477575.1' already exists. Skipping...
Folder 'GCA_045063145.1_PDT002477570.1' already exists. Skipping...
Folder 'GCA_045063205.1_PDT002477574.1' already exists. Skipping...
Folder 'GCA_045063225.1_PDT002477573.1' already exists. Skipping...
Folder 'GCA_045063265.1_PDT002477569.1' already exists. Skipping...
Folder 'GCA_045063305.1_PDT002477568.1' already exists. Skipping...
Folder 'GCA_045063325.1_PDT002477565.1' already exists. Skipping...
Folder 'GCA_045063345.1_PDT002477566.1' already 

Retreiveing data: 129519it [11:16:49,  3.08s/it]

Error downloading the file: ('Connection broken: IncompleteRead(81920 bytes read, 1906619 more expected)', IncompleteRead(81920 bytes read, 1906619 more expected))


Retreiveing data: 138288it [18:48:32,  2.43s/it]

In [ ]:
# END